# BERT Text Classification: Preprocessing Pipeline

## Objective
Prepare fake news detection datasets for BERT-based classification with lightweight, linguistically-aware preprocessing.

**Key Principle**: Preserve natural language structure for BERT's contextual understanding. No aggressive stemming, lemmatization, or stopword removal.

---

## Project Overview
- **Task**: Classification de fausses nouvelles en 3 classes
- **Model**: BERT (bert-base-uncased)
- **Datasets**: Train, Validation, Test (TSV format)
- **Preprocessing Philosophy**: Minimal but effective cleaning

## Section 1: Load Datasets from TSV Files

Load train, validation, and test datasets from TSV files in the data/ directory using pandas.

In [31]:
# Define paths to data files
DATA_PATH = Path("data")
train_path = DATA_PATH / "train.tsv"
valid_path = DATA_PATH / "valid.tsv"
test_path = DATA_PATH / "test.tsv"

# Define column names based on the TSV structure
# Columns: id, label, statement, subjects, speaker, job, state, party, count_support, count_refute, count_discuss, count_agree, count_disagree, context
column_names = ["id", "label", "statement", "subjects", "speaker", "job", "state", "party", 
                "count_support", "count_refute", "count_discuss", "count_agree", "count_disagree", "context"]

# Load datasets
df_train = pd.read_csv(train_path, sep="\t", names=column_names, dtype={"statement": str, "label": str})
df_valid = pd.read_csv(valid_path, sep="\t", names=column_names, dtype={"statement": str, "label": str})
df_test = pd.read_csv(test_path, sep="\t", names=column_names, dtype={"statement": str, "label": str})

print("✓ Datasets loaded successfully\n")
print(f"Training set shape: {df_train.shape}")
print(f"Validation set shape: {df_valid.shape}")
print(f"Test set shape: {df_test.shape}")

✓ Datasets loaded successfully

Training set shape: (10240, 14)
Validation set shape: (1284, 14)
Test set shape: (1267, 14)


## Section 2: Exploratory Data Analysis and Data Quality Checks

Examine dataset dimensions, column names, data types, and identify missing values.

In [32]:
# Display column information for training set
print("=" * 80)
print("TRAINING SET - BASIC INFORMATION")
print("=" * 80)
print(f"\nDataset Shape: {df_train.shape}\n")
print("Column Names and Data Types:")
print(df_train.dtypes)

print("\n" + "=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
missing_train = df_train.isnull().sum()
missing_valid = df_valid.isnull().sum()
missing_test = df_test.isnull().sum()

print("\nTraining Set Missing Values:")
print(missing_train[missing_train > 0])
print(f"Total rows: {len(df_train)}")

print("\nValidation Set Missing Values:")
print(missing_valid[missing_valid > 0] if missing_valid.sum() > 0 else "No missing values")
print(f"Total rows: {len(df_valid)}")

print("\nTest Set Missing Values:")
print(missing_test[missing_test > 0] if missing_test.sum() > 0 else "No missing values")
print(f"Total rows: {len(df_test)}")

# Display label distribution
print("\n" + "=" * 80)
print("LABEL DISTRIBUTION (BEFORE CONSOLIDATION)")
print("=" * 80)
print("\nTraining Set Labels:")
print(df_train["label"].value_counts())
print(f"\nValidation Set Labels:")
print(df_valid["label"].value_counts())
print(f"\nTest Set Labels:")
print(df_test["label"].value_counts())

# Display sample records
print("\n" + "=" * 80)
print("SAMPLE RECORDS FROM TRAINING SET")
print("=" * 80)
print("\nFirst 3 records (focusing on label and statement):")
for idx, (label, statement) in enumerate(zip(df_train["label"].head(3), df_train["statement"].head(3)), 1):
    print(f"\n{idx}. Label: {label}")
    print(f"   Statement: {statement[:100]}..." if len(str(statement)) > 100 else f"   Statement: {statement}")

TRAINING SET - BASIC INFORMATION

Dataset Shape: (10240, 14)

Column Names and Data Types:
id                 object
label              object
statement          object
subjects           object
speaker            object
job                object
state              object
party              object
count_support     float64
count_refute      float64
count_discuss     float64
count_agree       float64
count_disagree    float64
context            object
dtype: object

MISSING VALUES ANALYSIS

Training Set Missing Values:
subjects             2
speaker              2
job               2898
state             2210
party                2
count_support        2
count_refute         2
count_discuss        2
count_agree          2
count_disagree       2
context            102
dtype: int64
Total rows: 10240

Validation Set Missing Values:
job        345
state      279
context     12
dtype: int64
Total rows: 1284

Test Set Missing Values:
job        325
state      262
context     17
dtype: int64
T

## Section 3: Handle Missing Values

Identify and handle missing values appropriately based on their frequency and impact.

In [33]:
# We only need 'label' and 'statement' columns for BERT classification
# Keep these two columns and drop rows with missing values in either column

def handle_missing_values(df, dataset_name):
    """
    Remove rows with missing values in critical columns (label and statement).
    These are the only columns needed for BERT classification.
    """
    # Select only the relevant columns
    df_cleaned = df[["label", "statement"]].copy()
    
    # Count rows before cleaning
    rows_before = len(df_cleaned)
    
    # Drop rows with missing label or statement
    df_cleaned = df_cleaned.dropna(subset=["label", "statement"])
    
    rows_after = len(df_cleaned)
    rows_removed = rows_before - rows_after
    
    print(f"{dataset_name}:")
    print(f"  Rows before: {rows_before}")
    print(f"  Rows removed: {rows_removed}")
    print(f"  Rows after: {rows_after}")
    
    return df_cleaned

print("HANDLING MISSING VALUES")
print("=" * 80)
df_train = handle_missing_values(df_train, "Training Set")
print()
df_valid = handle_missing_values(df_valid, "Validation Set")
print()
df_test = handle_missing_values(df_test, "Test Set")

print("\n✓ Missing values handled")
print(f"\nFinal dataset shapes:")
print(f"  Training: {df_train.shape}")
print(f"  Validation: {df_valid.shape}")
print(f"  Test: {df_test.shape}")

HANDLING MISSING VALUES
Training Set:
  Rows before: 10240
  Rows removed: 0
  Rows after: 10240

Validation Set:
  Rows before: 1284
  Rows removed: 0
  Rows after: 1284

Test Set:
  Rows before: 1267
  Rows removed: 0
  Rows after: 1267

✓ Missing values handled

Final dataset shapes:
  Training: (10240, 2)
  Validation: (1284, 2)
  Test: (1267, 2)


## Section 4: Consolidate Labels into Three Classes

Map original labels to three consolidated classes:
- **"True"** ← [true, mostly-true]
- **"Nuanced"** ← [half-true, barely-true]
- **"False"** ← [false, pants-fire]

In [34]:
# Define label mapping: convert 6-class to 3-class classification
label_mapping = {
    "true": "True",
    "mostly-true": "True",
    "half-true": "Nuanced",
    "barely-true": "Nuanced",
    "false": "False",
    "pants-fire": "False"
}

def consolidate_labels(df):
    """Map original labels to consolidated 3-class labels."""
    df = df.copy()
    df["label"] = df["label"].str.lower().map(label_mapping)
    return df

# Apply label consolidation
print("=" * 80)
print("LABEL CONSOLIDATION: 6 CLASSES → 3 CLASSES")
print("=" * 80)

df_train = consolidate_labels(df_train)
df_valid = consolidate_labels(df_valid)
df_test = consolidate_labels(df_test)

# Verify consolidation
print("\nTraining Set - Consolidated Labels:")
print(df_train["label"].value_counts())
print(f"\nValidation Set - Consolidated Labels:")
print(df_valid["label"].value_counts())
print(f"\nTest Set - Consolidated Labels:")
print(df_test["label"].value_counts())

print("\n✓ Labels consolidated successfully")
print(f"\nLabel Distribution:")
print(f"  Training: {dict(df_train['label'].value_counts())}")
print(f"  Validation: {dict(df_valid['label'].value_counts())}")
print(f"  Test: {dict(df_test['label'].value_counts())}")

LABEL CONSOLIDATION: 6 CLASSES → 3 CLASSES

Training Set - Consolidated Labels:
label
Nuanced    3768
True       3638
False      2834
Name: count, dtype: int64

Validation Set - Consolidated Labels:
label
Nuanced    485
True       420
False      379
Name: count, dtype: int64

Test Set - Consolidated Labels:
label
Nuanced    477
True       449
False      341
Name: count, dtype: int64

✓ Labels consolidated successfully

Label Distribution:
  Training: {'Nuanced': 3768, 'True': 3638, 'False': 2834}
  Validation: {'Nuanced': 485, 'True': 420, 'False': 379}
  Test: {'Nuanced': 477, 'True': 449, 'False': 341}


## Section 5: Apply Lightweight Text Cleaning

Implement gentle text cleaning optimized for BERT:
- Convert to lowercase
- Remove multiple consecutive spaces
- Preserve meaningful punctuation
- Maintain word structure (no stemming/lemmatization)
- Minimal special character removal

In [35]:
def clean_text_for_bert(text):
    """
    Apply lightweight text cleaning suitable for BERT preprocessing.
    
    Principles:
    - Preserve natural language structure
    - NO stemming or lemmatization
    - NO stopword removal
    - Minimal special character handling
    - Keep meaningful punctuation
    
    Operations:
    1. Convert to lowercase (BERT expects lowercase for bert-base-uncased)
    2. Remove extra whitespace (multiple spaces, tabs, newlines)
    3. Keep punctuation as BERT tokenizer handles it well
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Replace multiple whitespace characters with single space
    # This includes spaces, tabs, newlines, etc.
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading and trailing whitespace
    text = text.strip()
    
    return text

# Store original text for comparison
df_train["statement_original"] = df_train["statement"].copy()
df_valid["statement_original"] = df_valid["statement"].copy()
df_test["statement_original"] = df_test["statement"].copy()

# Apply cleaning
print("=" * 80)
print("APPLYING LIGHTWEIGHT TEXT CLEANING")
print("=" * 80)

df_train["statement"] = df_train["statement"].apply(clean_text_for_bert)
df_valid["statement"] = df_valid["statement"].apply(clean_text_for_bert)
df_test["statement"] = df_test["statement"].apply(clean_text_for_bert)

print("\n✓ Text cleaning applied to all datasets")
print(f"  Training: {len(df_train)} samples cleaned")
print(f"  Validation: {len(df_valid)} samples cleaned")
print(f"  Test: {len(df_test)} samples cleaned")

APPLYING LIGHTWEIGHT TEXT CLEANING

✓ Text cleaning applied to all datasets
  Training: 10240 samples cleaned
  Validation: 1284 samples cleaned
  Test: 1267 samples cleaned


## Section 6: Display Before and After Preprocessing Examples

Show sample text examples before and after cleaning to verify preprocessing preserves natural language.

In [36]:
print("=" * 80)
print("BEFORE / AFTER PREPROCESSING EXAMPLES")
print("=" * 80)

# Select diverse examples from training set
sample_indices = [0, 10, 50, 100, 200]

for i, idx in enumerate(sample_indices, 1):
    if idx < len(df_train):
        print(f"\n{'─' * 80}")
        print(f"EXAMPLE {i} - Label: {df_train.iloc[idx]['label']}")
        print(f"{'─' * 80}")
        
        original = df_train.iloc[idx]['statement_original']
        cleaned = df_train.iloc[idx]['statement']
        
        print(f"\nBEFORE:")
        print(f"  {original[:120]}..." if len(str(original)) > 120 else f"  {original}")
        
        print(f"\nAFTER:")
        print(f"  {cleaned[:120]}..." if len(str(cleaned)) > 120 else f"  {cleaned}")
        
        # Show what changed
        if original.lower() != cleaned:
            print(f"\n  ✓ Changes detected (whitespace normalization)")
        else:
            print(f"\n  ✓ Text unchanged (already clean)")

print(f"\n{'═' * 80}")

BEFORE / AFTER PREPROCESSING EXAMPLES

────────────────────────────────────────────────────────────────────────────────
EXAMPLE 1 - Label: False
────────────────────────────────────────────────────────────────────────────────

BEFORE:
  Says the Annies List political group supports third-trimester abortions on demand.

AFTER:
  says the annies list political group supports third-trimester abortions on demand.

  ✓ Text unchanged (already clean)

────────────────────────────────────────────────────────────────────────────────
EXAMPLE 2 - Label: True
────────────────────────────────────────────────────────────────────────────────

BEFORE:
  For the first time in history, the share of the national popular vote margin is smaller than the Latino vote margin.

AFTER:
  for the first time in history, the share of the national popular vote margin is smaller than the latino vote margin.

  ✓ Text unchanged (already clean)

────────────────────────────────────────────────────────────────────────

## Section 7: Encode Labels to Numeric Format

Convert string labels to numeric format for compatibility with model training.

In [37]:
# Create label encoder
label_encoder = LabelEncoder()

# Fit encoder on all unique labels (ensuring consistent encoding across splits)
unique_labels = pd.concat([df_train["label"], df_valid["label"], df_test["label"]]).unique()
label_encoder.fit(unique_labels)

# Store label mapping for reference
label_to_id = {label: idx for idx, label in enumerate(label_encoder.classes_)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print("=" * 80)
print("LABEL ENCODING")
print("=" * 80)
print(f"\nLabel to ID Mapping:")
for label, id_val in sorted(label_to_id.items()):
    print(f"  '{label}' → {id_val}")

# Encode labels
df_train["label_encoded"] = label_encoder.transform(df_train["label"])
df_valid["label_encoded"] = label_encoder.transform(df_valid["label"])
df_test["label_encoded"] = label_encoder.transform(df_test["label"])

print(f"\n✓ Labels encoded successfully")
print(f"\nTraining Set - Label Distribution (Numeric):")
print(df_train["label_encoded"].value_counts().sort_index())
print(f"\nValidation Set - Label Distribution (Numeric):")
print(df_valid["label_encoded"].value_counts().sort_index())
print(f"\nTest Set - Label Distribution (Numeric):")
print(df_test["label_encoded"].value_counts().sort_index())

LABEL ENCODING

Label to ID Mapping:
  'False' → 0
  'Nuanced' → 1
  'True' → 2

✓ Labels encoded successfully

Training Set - Label Distribution (Numeric):
label_encoded
0    2834
1    3768
2    3638
Name: count, dtype: int64

Validation Set - Label Distribution (Numeric):
label_encoded
0    379
1    485
2    420
Name: count, dtype: int64

Test Set - Label Distribution (Numeric):
label_encoded
0    341
1    477
2    449
Name: count, dtype: int64


## Section 8: Tokenize Text Data with HuggingFace BERT Tokenizer

Load bert-base-uncased tokenizer and tokenize all text samples with appropriate parameters.

In [44]:
# Load BERT tokenizer
print("=" * 80)
print("LOADING BERT TOKENIZER")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"✓ Tokenizer loaded successfully")
print(f"  Model: bert-base-uncased")
print(f"  Vocabulary size: {len(tokenizer)}")
print(f"  Max sequence length: 512 (BERT standard)")

# Define maximum sequence length for BERT
# BERT's maximum is 512, but we'll use a practical limit
MAX_LENGTH = 128  # Adjust based on dataset characteristics

# Analyze token length distribution
print(f"\n" + "=" * 80)
print("ANALYZING TOKEN LENGTH DISTRIBUTION")
print("=" * 80)

def get_token_count(text):
    """Get the number of tokens for a text sample."""
    return len(tokenizer.encode(text, add_special_tokens=True))

# Sample analysis on first 500 training examples
sample_size = min(500, len(df_train))
token_counts = [get_token_count(text) for text in df_train["statement"].iloc[:sample_size]]

print(f"\nAnalyzing {sample_size} training samples...")
print(f"  Min tokens: {min(token_counts)}")
print(f"  Max tokens: {max(token_counts)}")
print(f"  Mean tokens: {np.mean(token_counts):.2f}")
print(f"  Median tokens: {np.median(token_counts):.2f}")
print(f"  Std dev: {np.std(token_counts):.2f}")

# Check how many samples exceed MAX_LENGTH
exceeds_max = sum(1 for count in token_counts if count > MAX_LENGTH)
print(f"\nSamples exceeding {MAX_LENGTH} tokens: {exceeds_max} ({100*exceeds_max/sample_size:.2f}%)")

LOADING BERT TOKENIZER
✓ Tokenizer loaded successfully
  Model: bert-base-uncased
  Vocabulary size: 30522
  Max sequence length: 512 (BERT standard)

ANALYZING TOKEN LENGTH DISTRIBUTION

Analyzing 500 training samples...
  Min tokens: 6
  Max tokens: 75
  Mean tokens: 24.77
  Median tokens: 24.00
  Std dev: 9.30

Samples exceeding 128 tokens: 0 (0.00%)


In [39]:
# Tokenize all datasets
print(f"\n" + "=" * 80)
print("TOKENIZING DATASETS")
print("=" * 80)

def tokenize_dataset(texts, labels, dataset_name):
    """
    Tokenize a dataset and return input_ids, attention_masks, and labels.
    
    Parameters:
    - texts: list of text samples
    - labels: list of encoded labels
    - dataset_name: name of the dataset for logging
    
    Returns:
    - Dictionary with input_ids, attention_mask, and labels
    """
    print(f"\nTokenizing {dataset_name} ({len(texts)} samples)...")
    
    encodings = tokenizer(
        list(texts),
        max_length=MAX_LENGTH,
        padding='max_length',  # Pad all to MAX_LENGTH
        truncation=True,       # Truncate if longer than MAX_LENGTH
        return_tensors=None    # Return lists instead of torch tensors (more flexible)
    )
    
    tokenized_data = {
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': list(labels)
    }
    
    print(f"  ✓ Tokenization complete")
    print(f"    Input IDs shape: {np.array(tokenized_data['input_ids']).shape}")
    print(f"    Attention masks shape: {np.array(tokenized_data['attention_mask']).shape}")
    
    return tokenized_data

# Tokenize all three datasets
train_tokenized = tokenize_dataset(df_train["statement"], df_train["label_encoded"], "Training Set")
valid_tokenized = tokenize_dataset(df_valid["statement"], df_valid["label_encoded"], "Validation Set")
test_tokenized = tokenize_dataset(df_test["statement"], df_test["label_encoded"], "Test Set")

print(f"\n✓ All datasets tokenized successfully")


TOKENIZING DATASETS

Tokenizing Training Set (10240 samples)...
  ✓ Tokenization complete
    Input IDs shape: (10240, 128)
    Attention masks shape: (10240, 128)

Tokenizing Validation Set (1284 samples)...
  ✓ Tokenization complete
    Input IDs shape: (1284, 128)
    Attention masks shape: (1284, 128)

Tokenizing Test Set (1267 samples)...
  ✓ Tokenization complete
    Input IDs shape: (1267, 128)
    Attention masks shape: (1267, 128)

✓ All datasets tokenized successfully


## Section 9: Generate Attention Masks and Prepare Features

Verify padding, truncation, and prepare final feature arrays for model input.

In [40]:
print("=" * 80)
print("VERIFYING TOKENIZED DATA STRUCTURE")
print("=" * 80)

# Verify structure and content
print(f"\nTraining Set Tokenized Data:")
print(f"  Type: {type(train_tokenized)}")
print(f"  Keys: {train_tokenized.keys()}")
print(f"  input_ids: {len(train_tokenized['input_ids'])} samples")
print(f"  attention_mask: {len(train_tokenized['attention_mask'])} samples")
print(f"  labels: {len(train_tokenized['labels'])} samples")

# Display sample tokenized record
print(f"\nSample Tokenized Record (Training Set, Index 0):")
sample_idx = 0
print(f"  Original text: {df_train['statement'].iloc[sample_idx][:60]}...")
print(f"  Label (numeric): {train_tokenized['labels'][sample_idx]}")
print(f"  Label (string): {id_to_label[train_tokenized['labels'][sample_idx]]}")
print(f"  Input IDs (first 20 tokens): {train_tokenized['input_ids'][sample_idx][:20]}")
print(f"  Attention mask (first 20): {train_tokenized['attention_mask'][sample_idx][:20]}")
print(f"  Sequence length: {len(train_tokenized['input_ids'][sample_idx])}")

# Verify all sequences have same length
print(f"\nVerifying Sequence Length Consistency:")
train_lengths = set(len(seq) for seq in train_tokenized['input_ids'])
valid_lengths = set(len(seq) for seq in valid_tokenized['input_ids'])
test_lengths = set(len(seq) for seq in test_tokenized['input_ids'])

print(f"  Training set lengths: {train_lengths}")
print(f"  Validation set lengths: {valid_lengths}")
print(f"  Test set lengths: {test_lengths}")
print(f"  ✓ All sequences properly padded to {MAX_LENGTH}")

# Convert to numpy arrays for efficient storage and processing
print(f"\n" + "=" * 80)
print("CONVERTING TO NUMPY ARRAYS")
print("=" * 80)

train_tokenized['input_ids'] = np.array(train_tokenized['input_ids'])
train_tokenized['attention_mask'] = np.array(train_tokenized['attention_mask'])
train_tokenized['labels'] = np.array(train_tokenized['labels'])

valid_tokenized['input_ids'] = np.array(valid_tokenized['input_ids'])
valid_tokenized['attention_mask'] = np.array(valid_tokenized['attention_mask'])
valid_tokenized['labels'] = np.array(valid_tokenized['labels'])

test_tokenized['input_ids'] = np.array(test_tokenized['input_ids'])
test_tokenized['attention_mask'] = np.array(test_tokenized['attention_mask'])
test_tokenized['labels'] = np.array(test_tokenized['labels'])

print(f"\n✓ Converted to numpy arrays:")
print(f"  Training - input_ids: {train_tokenized['input_ids'].shape}, dtype: {train_tokenized['input_ids'].dtype}")
print(f"  Training - attention_mask: {train_tokenized['attention_mask'].shape}, dtype: {train_tokenized['attention_mask'].dtype}")
print(f"  Training - labels: {train_tokenized['labels'].shape}, dtype: {train_tokenized['labels'].dtype}")

VERIFYING TOKENIZED DATA STRUCTURE

Training Set Tokenized Data:
  Type: <class 'dict'>
  Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
  input_ids: 10240 samples
  attention_mask: 10240 samples
  labels: 10240 samples

Sample Tokenized Record (Training Set, Index 0):
  Original text: says the annies list political group supports third-trimeste...
  Label (numeric): 0
  Label (string): False
  Input IDs (first 20 tokens): [101, 2758, 1996, 8194, 2015, 2862, 2576, 2177, 6753, 2353, 1011, 12241, 20367, 11324, 2015, 2006, 5157, 1012, 102, 0]
  Attention mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
  Sequence length: 128

Verifying Sequence Length Consistency:
  Training set lengths: {128}
  Validation set lengths: {128}
  Test set lengths: {128}
  ✓ All sequences properly padded to 128

CONVERTING TO NUMPY ARRAYS

✓ Converted to numpy arrays:
  Training - input_ids: (10240, 128), dtype: int32
  Training - attention_mask: (10240, 128), dtype: i

## Section 10: Verify Data Leakage Between Train/Validation/Test Splits

Check for duplicate samples and ensure no overlap between train, validation, and test sets.

In [41]:
def find_and_display_duplicates(df, split_name, max_display=10):
    """Find and display duplicate samples in a split."""
    duplicates_dict = {}
    
    # Find duplicates
    for idx, text in enumerate(df["statement"]):
        if text in duplicates_dict:
            duplicates_dict[text].append(idx)
        else:
            duplicates_dict[text] = [idx]
    
    # Filter only duplicates (texts that appear more than once)
    actual_duplicates = {text: indices for text, indices in duplicates_dict.items() if len(indices) > 1}
    
    if actual_duplicates:
        print(f"\n{split_name} - {len(actual_duplicates)} duplicate(s) found:")
        for i, (text, indices) in enumerate(list(actual_duplicates.items())[:max_display], 1):
            print(f"\n  {'=' * 76}")
            print(f"  DUPLICATE #{i}:")
            print(f"  {'=' * 76}")
            print(f"\n  FULL TEXT:")
            print(f"  {text}")
            print(f"\n  Appears {len(indices)} times at indices: {indices}")
            print(f"  Label(s): {[df.iloc[idx]['label'] for idx in indices]}")
    else:
        print(f"\n{split_name}: ✓ No duplicates found")

## Section 11: Display Final Preprocessed Dataset Dimensions

Summary of final preprocessed datasets ready for BERT model training.

In [42]:
print("=" * 80)
print("FINAL PREPROCESSED DATASETS SUMMARY")
print("=" * 80)

# Summary table
summary_data = {
    "Dataset": ["Training", "Validation", "Test"],
    "Samples": [len(train_tokenized['labels']), len(valid_tokenized['labels']), len(test_tokenized['labels'])],
    "input_ids shape": [str(train_tokenized['input_ids'].shape), 
                        str(valid_tokenized['input_ids'].shape),
                        str(test_tokenized['input_ids'].shape)],
    "attention_mask shape": [str(train_tokenized['attention_mask'].shape),
                             str(valid_tokenized['attention_mask'].shape),
                             str(test_tokenized['attention_mask'].shape)],
    "labels shape": [str(train_tokenized['labels'].shape),
                     str(valid_tokenized['labels'].shape),
                     str(test_tokenized['labels'].shape)]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Detailed statistics
print("\n\n" + "=" * 80)
print("DETAILED LABEL DISTRIBUTION (FINAL)")
print("=" * 80)

def print_label_distribution(labels, split_name):
    """Print label distribution for a split."""
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\n{split_name}:")
    total = len(labels)
    for label_id, count in zip(unique, counts):
        percentage = 100 * count / total
        label_name = id_to_label[label_id]
        print(f"  {label_name} ({label_id}): {count:5d} samples ({percentage:6.2f}%)")
    print(f"  {'Total':<30}: {total:5d} samples")

print_label_distribution(train_tokenized['labels'], "Training Set")
print_label_distribution(valid_tokenized['labels'], "Validation Set")
print_label_distribution(test_tokenized['labels'], "Test Set")

# Data type information
print("\n" + "=" * 80)
print("DATA TYPES AND MEMORY")
print("=" * 80)

train_memory = (train_tokenized['input_ids'].nbytes + 
                train_tokenized['attention_mask'].nbytes + 
                train_tokenized['labels'].nbytes) / (1024**2)

valid_memory = (valid_tokenized['input_ids'].nbytes + 
                valid_tokenized['attention_mask'].nbytes + 
                valid_tokenized['labels'].nbytes) / (1024**2)

test_memory = (test_tokenized['input_ids'].nbytes + 
               test_tokenized['attention_mask'].nbytes + 
               test_tokenized['labels'].nbytes) / (1024**2)

print(f"\nTraining Set:")
print(f"  input_ids dtype: {train_tokenized['input_ids'].dtype}, memory: {train_memory:.2f} MB")
print(f"  attention_mask dtype: {train_tokenized['attention_mask'].dtype}")
print(f"  labels dtype: {train_tokenized['labels'].dtype}")

print(f"\nValidation Set:")
print(f"  input_ids dtype: {valid_tokenized['input_ids'].dtype}, memory: {valid_memory:.2f} MB")
print(f"  attention_mask dtype: {valid_tokenized['attention_mask'].dtype}")
print(f"  labels dtype: {valid_tokenized['labels'].dtype}")

print(f"\nTest Set:")
print(f"  input_ids dtype: {test_tokenized['input_ids'].dtype}, memory: {test_memory:.2f} MB")
print(f"  attention_mask dtype: {test_tokenized['attention_mask'].dtype}")
print(f"  labels dtype: {test_tokenized['labels'].dtype}")

# Tokenization statistics
print("\n" + "=" * 80)
print("TOKENIZATION PARAMETERS")
print("=" * 80)
print(f"\n  Max sequence length: {MAX_LENGTH}")
print(f"  Padding strategy: max_length (pad all sequences to {MAX_LENGTH})")
print(f"  Truncation strategy: enabled (truncate sequences > {MAX_LENGTH})")
print(f"  Tokenizer: bert-base-uncased")
print(f"  Vocabulary size: {len(tokenizer)}")

print("\n" + "=" * 80)
print("✓ PREPROCESSING PIPELINE COMPLETE")
print("=" * 80)
print(f"\nDatasets are ready for BERT model training!")
print(f"Total samples processed: {len(train_tokenized['labels']) + len(valid_tokenized['labels']) + len(test_tokenized['labels'])}")
print(f"Total memory used: {train_memory + valid_memory + test_memory:.2f} MB")

FINAL PREPROCESSED DATASETS SUMMARY

   Dataset  Samples input_ids shape attention_mask shape labels shape
  Training    10240    (10240, 128)         (10240, 128)     (10240,)
Validation     1284     (1284, 128)          (1284, 128)      (1284,)
      Test     1267     (1267, 128)          (1267, 128)      (1267,)


DETAILED LABEL DISTRIBUTION (FINAL)

Training Set:
  False (0):  2834 samples ( 27.68%)
  Nuanced (1):  3768 samples ( 36.80%)
  True (2):  3638 samples ( 35.53%)
  Total                         : 10240 samples

Validation Set:
  False (0):   379 samples ( 29.52%)
  Nuanced (1):   485 samples ( 37.77%)
  True (2):   420 samples ( 32.71%)
  Total                         :  1284 samples

Test Set:
  False (0):   341 samples ( 26.91%)
  Nuanced (1):   477 samples ( 37.65%)
  True (2):   449 samples ( 35.44%)
  Total                         :  1267 samples

DATA TYPES AND MEMORY

Training Set:
  input_ids dtype: int32, memory: 10.04 MB
  attention_mask dtype: int32
  labels dt

---

## Appendix: Saving Preprocessed Data

Optional: Save the preprocessed datasets for later use in model training.

In [43]:
# Optional: Save preprocessed data for later use
import pickle

# Create output directory
output_dir = Path("preprocessed_data")
output_dir.mkdir(exist_ok=True)

# Save tokenized datasets
print("=" * 80)
print("SAVING PREPROCESSED DATA")
print("=" * 80)

# Save as pickle files for easy loading
with open(output_dir / "train_tokenized.pkl", "wb") as f:
    pickle.dump(train_tokenized, f)
    print("✓ Training data saved: preprocessed_data/train_tokenized.pkl")

with open(output_dir / "valid_tokenized.pkl", "wb") as f:
    pickle.dump(valid_tokenized, f)
    print("✓ Validation data saved: preprocessed_data/valid_tokenized.pkl")

with open(output_dir / "test_tokenized.pkl", "wb") as f:
    pickle.dump(test_tokenized, f)
    print("✓ Test data saved: preprocessed_data/test_tokenized.pkl")

# Save label encoder and mappings
with open(output_dir / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)
    print("✓ Label encoder saved: preprocessed_data/label_encoder.pkl")

# Save label mappings as JSON for reference
import json
mappings = {
    "label_to_id": label_to_id,
    "id_to_label": {str(k): v for k, v in id_to_label.items()},
    "max_length": MAX_LENGTH
}
with open(output_dir / "label_mappings.json", "w") as f:
    json.dump(mappings, f, indent=2)
    print("✓ Label mappings saved: preprocessed_data/label_mappings.json")

print(f"\n✓ All preprocessed data saved to {output_dir}/")
print("\nTo load data in training script:")
print("""
import pickle
with open('preprocessed_data/train_tokenized.pkl', 'rb') as f:
    train_tokenized = pickle.load(f)
""")


SAVING PREPROCESSED DATA
✓ Training data saved: preprocessed_data/train_tokenized.pkl
✓ Validation data saved: preprocessed_data/valid_tokenized.pkl
✓ Test data saved: preprocessed_data/test_tokenized.pkl
✓ Label encoder saved: preprocessed_data/label_encoder.pkl
✓ Label mappings saved: preprocessed_data/label_mappings.json

✓ All preprocessed data saved to preprocessed_data/

To load data in training script:

import pickle
with open('preprocessed_data/train_tokenized.pkl', 'rb') as f:
    train_tokenized = pickle.load(f)

